# Lesson 13 Lab — Chunked Prefill and Decode Interference

**Puzzle:** Should a long prompt monopolize one scheduling iteration while short requests wait?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Large Prefill batches improve compute utilization but can delay active Decode sequences. Chunking divides prompt work into token budgets so the scheduler can interleave it with latency-sensitive generation.


## 0. Predict before running

1. Predict short-request maximum delay without chunking.
2. Calculate the number of 512-token chunks.
3. Name the native trace needed to choose the budget.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The lab replays a measured-cost scheduling model and probes installed chunked-prefill arguments. It compares an unchunked 4096-token prompt with 512-token chunks while short Decode jobs arrive.

- Chunking changes when work is scheduled, not total prompt tokens.
- Smaller chunks can lower blocking time while adding launch/scheduling overhead.
- TTFT and ITL may move in opposite directions.


## 2. Derive the mechanism

A scheduler with maximum batched-token budget `B` can consume a prompt in `ceil(L/B)` chunks. Smaller chunks create more scheduling opportunities for Decode but may add overhead and reduce Prefill efficiency. The correct chunk size is therefore an SLO trade-off, not a universal minimum.

### Mechanism at a glance

```mermaid
gantt
  title Mixed prompt and Decode work
  dateFormat X
  axisFormat %L
  section Unchunked
  Long Prefill :0, 8
  Short Decode :8, 11
  section Chunked
  Prefill chunk 1 :0, 2
  Short Decode :2, 3
  Prefill chunk 2 :3, 5
```

### Walk it step by step

1. **Set a token budget.** Bound how much prompt work enters one scheduler iteration.
2. **Split the long prompt.** Create multiple resumable Prefill chunks.
3. **Admit Decode between chunks.** Give active requests opportunities to advance.
4. **Sweep the trade-off.** Measure TTFT, ITL, and throughput together on the native engine.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 13
LESSON_TITLE = 'Chunked Prefill and Decode Interference'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260825
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | one monolithic long Prefill |
| Candidate | fixed-size chunked Prefill interleaved with Decode |
| Held constant | token demands, per-token cost assumptions, arrival times, priority, and GPU identity |
| Measurements | long TTFT proxy, short p95 delay, scheduling rounds, and CLI feature presence |
| Evidence | `numerical-model` |

**Experiment:** Interleave one long Prefill with short Decode jobs in a cost-calibrated scheduler model and inspect CLI support.


## 5. Inspect the experiment code

The simulation exposes its cost coefficients and full event timeline. It remains separate from native vLLM timing because one GPU kernel does not have constant per-token cost.

Do not execute until the code matches the frozen table.


In [2]:
long_tokens=4096; chunk_tokens=512; jobs=[{"arrival":1.,"steps":8},{"arrival":2.,"steps":6},
    {"arrival":3.,"steps":4},{"arrival":5.,"steps":3}]
prefill_cost=.002; decode_cost=.22; overhead=.08
def mixed(chunked):
    now=0.; waits=[]; chunks=0; remaining=long_tokens; pending=[dict(x) for x in jobs]
    while remaining or pending:
        if chunked and pending and pending[0]["arrival"]<=now:
            job=pending.pop(0); waits.append(now-job["arrival"]); now+=job["steps"]*decode_cost
        elif remaining:
            take=min(chunk_tokens if chunked else remaining,remaining); now+=take*prefill_cost+(overhead if chunked else 0)
            remaining-=take; chunks+=1
        else:
            job=pending.pop(0); now=max(now,job["arrival"]); waits.append(now-job["arrival"]); now+=job["steps"]*decode_cost
    return {"long_finish":now,"short_p95_delay":percentile(waits,.95),"short_max_delay":max(waits),
            "prefill_chunks":chunks,"waits":waits}
_,help_text=cli_help("serve")
metrics={"unchunked":mixed(False),"chunked":mixed(True),
         "cli":{"chunked_prefill":"--enable-chunked-prefill" in help_text,
                "max_num_batched_tokens":"--max-num-batched-tokens" in help_text},
         "cost_assumptions":{"prefill_per_token":prefill_cost,"decode_step":decode_cost,"chunk_overhead":overhead}}
analysis=(f"512-token chunking created {metrics['chunked']['prefill_chunks']} chunks and changed "
          f"short-job p95 delay from {metrics['unchunked']['short_p95_delay']:.3f} to "
          f"{metrics['chunked']['short_p95_delay']:.3f} modeled units. Native traffic is still required.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Unchunked short p95 | 8.224000 |
| Chunked short p95 | 1.136000 |
| Unchunked long finish | 12.812000 |
| Chunked long finish | 13.452000 |
| Chunks | 8 |
| CLI support | no |


## 7. Explain the result

512-token chunking created 8 chunks and changed short-job p95 delay from 8.224 to 1.136 modeled units. Native traffic is still required.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. A transparent allocator, scheduler, gateway, or policy model executed. It establishes the stated invariant, not native vLLM performance.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 13, "title": 'Chunked Prefill and Decode Interference', "environment": ENV,
    "evidence_label": 'numerical-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Chunked Prefill creates scheduling opportunities; its production value must be selected from native mixed-traffic trade-offs.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 13,
  "title": "Chunked Prefill and Decode Interference",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260825
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "unchunked": {
      "long_finish": 12.812000000000001,
      "short_p95_delay": 8.224,
      "short_max_delay": 8.272,
      "prefill_chunks": 1,
      "waits": [
        7.192,
        7.952,
        8.272,
        7.152000000000001
      ]
    },
    "chunked": {
      "long_finish": 13.451999999999998,
      "short_p95_delay": 1.1360000000000001,
      "short_max_delay": 1.1840000000000002,
      "prefill_chunks": 8,
      "waits": [
        0.10400000000000009,
        0.8639999999999999,
        1.1840000000000002,
        0.06400000000000006
      ]
    },
    "cli": {
      "chunked_prefill":

## 9. Make the bounded decision

> Chunked Prefill creates scheduling opportunities; its production value must be selected from native mixed-traffic trade-offs.

**Acceptance/rollback:** Choose a chunk budget only when native mixed-traffic replay meets both long-prompt TTFT and active-request ITL gates.

**Failure analysis:** Attention kernels scale nonlinearly with sequence length; CUDA graphs, batching, prefix hits, and compilation change step duration. Model results are directional only.


## 10. Extend the evidence

Sweep `max_num_batched_tokens` on the real engine with concurrent streaming clients and retain scheduler metrics, TTFT, ITL, throughput, and GPU utilization.

The full boundary and references are in [`README.md`](README.md).
